![Kayak](https://seekvectorlogo.com/wp-content/uploads/2018/01/kayak-vector-logo.png)

# Plan your trip with Kayak

## Contexte

Kayak souhaite proposer à ses utilisateurs une application de recommandation de destinations en France. Une étude interne révèle que **70% des utilisateurs souhaitent davantage d'informations** sur les destinations qu'ils envisagent avant de réserver.

L'objectif de ce projet est de construire un **pipeline de données complet** pour collecter, stocker et visualiser les données météo et hôtelières de 35 villes françaises.

## Architecture du pipeline

```
Sources (APIs + Scraping)
        ↓
   Data Lake (AWS S3)       ← stockage brut des CSV
        ↓
   ETL (Pandas + SQLAlchemy) ← nettoyage et transformation
        ↓
   Data Warehouse (AWS RDS MySQL) ← 3 tables relationnelles
        ↓
   Visualisations (Plotly)  ← cartes interactives
```

## Stack technique

| Étape | Outil |
|-------|-------|
| Géocodage | Nominatim (OpenStreetMap) |
| Météo | OpenWeatherMap API |
| Hôtels | Overpass API (OpenStreetMap) |
| Data Lake | AWS S3 |
| Data Warehouse | AWS RDS MySQL |
| Visualisation | Plotly |


## 0. Installation et imports


In [3]:
!pip install -q boto3 sqlalchemy pymysql plotly requests pandas

import requests
import pandas as pd
import numpy as np
import time
import os
import io
import boto3
from sqlalchemy import create_engine, text
import plotly.express as px

# Récupération des credentials depuis les Secrets Colab
# Pour configurer : menu Colab > Secrets > ajouter OWM_API_KEY, AWS_ACCESS_KEY, etc.
try:
    from google.colab import userdata
    OWM_API_KEY   = userdata.get('OWM_API_KEY')
    AWS_KEY       = userdata.get('AWS_ACCESS_KEY_ID')
    AWS_SECRET    = userdata.get('AWS_SECRET_ACCESS_KEY')
    RDS_PASSWORD  = userdata.get('RDS_PASSWORD')
    print('Secrets Colab chargés OK')
except Exception:
    # Fallback variables d'environnement (local)
    OWM_API_KEY  = os.environ.get('OWM_API_KEY', 'YOUR_KEY')
    AWS_KEY      = os.environ.get('AWS_ACCESS_KEY_ID', 'YOUR_KEY')
    AWS_SECRET   = os.environ.get('AWS_SECRET_ACCESS_KEY', 'YOUR_SECRET')
    RDS_PASSWORD = os.environ.get('RDS_PASSWORD', 'YOUR_PASSWORD')
    print('Variables d\'environnement chargées')

BUCKET_NAME = 'kayak-project-marine'
RDS_HOST    = 'kayak-db.c9eaa6c44d7i.eu-west-3.rds.amazonaws.com'
RDS_PORT    = 3306
RDS_USER    = 'admin'
RDS_DB      = 'kayak'

print('Imports OK')


Secrets Colab chargés OK
Imports OK


## 1. Collecte des coordonnées GPS Nominatim

**Nominatim** est un service de géocodage gratuit basé sur OpenStreetMap. Il convertit un nom de ville en coordonnées GPS (latitude, longitude).

Points importants :
- Le `User-Agent` est **obligatoire** dans les headers (politique d'utilisation de l'API)
- On respecte le **rate limit** d'1 requête/seconde avec `time.sleep(1)`
- On ajoute `, France` dans la requête pour lever l'ambiguïté géographique


In [4]:
cities = [
    'Mont Saint Michel', 'St Malo', 'Bayeux', 'Le Havre', 'Rouen',
    'Paris', 'Amiens', 'Lille', 'Strasbourg', 'Chateau du Haut Koenigsbourg',
    'Colmar', 'Eguisheim', 'Besancon', 'Dijon', 'Annecy', 'Grenoble',
    'Lyon', 'Gorges du Verdon', 'Bormes les Mimosas', 'Cassis',
    'Marseille', 'Aix en Provence', 'Avignon', 'Uzes', 'Nimes',
    'Aigues Mortes', 'Saintes Maries de la mer', 'Collioure', 'Carcassonne',
    'Ariege', 'Toulouse', 'Montauban', 'Biarritz', 'Bayonne', 'La Rochelle'
]

def get_gps(city_name):
    url = 'https://nominatim.openstreetmap.org/search'
    params = {'q': city_name + ', France', 'format': 'json', 'limit': 1}
    headers = {'User-Agent': 'kayak-project/1.0 (formation jedha cdsd)'}
    response = requests.get(url, params=params, headers=headers)
    data = response.json()
    if data:
        return float(data[0]['lat']), float(data[0]['lon'])
    return None, None

results = []
print('Collecte des coordonnées GPS...')
for i, city in enumerate(cities):
    lat, lon = get_gps(city)
    results.append({'id': i + 1, 'city': city, 'latitude': lat, 'longitude': lon})
    print(f'  {city} → ({lat}, {lon})')
    time.sleep(1)

df_cities = pd.DataFrame(results)
df_cities.dropna(subset=['latitude', 'longitude'], inplace=True)
df_cities.to_csv('cities_gps.csv', index=False)

print(f'\n{len(df_cities)} villes géocodées')
df_cities.head(10)


Collecte des coordonnées GPS...
  Mont Saint Michel → (48.6359541, -1.51146)
  St Malo → (48.649518, -2.0260409)
  Bayeux → (49.2764624, -0.7024738)
  Le Havre → (49.4938975, 0.1079732)
  Rouen → (49.4404591, 1.0939658)
  Paris → (48.8534951, 2.3483915)
  Amiens → (49.8941708, 2.2956951)
  Lille → (50.6365654, 3.0635282)
  Strasbourg → (48.584614, 7.7507127)
  Chateau du Haut Koenigsbourg → (48.249382, 7.3439412)
  Colmar → (48.0777517, 7.3579641)
  Eguisheim → (48.0447968, 7.3079618)
  Besancon → (47.2380222, 6.0243622)
  Dijon → (47.3215806, 5.0414701)
  Annecy → (45.8992348, 6.1288847)
  Grenoble → (45.1875602, 5.7357819)
  Lyon → (45.7578137, 4.8320114)
  Gorges du Verdon → (43.7496562, 6.3285616)
  Bormes les Mimosas → (43.1506968, 6.3419285)
  Cassis → (43.2140359, 5.5396318)
  Marseille → (43.2961743, 5.3699525)
  Aix en Provence → (43.5298424, 5.4474738)
  Avignon → (43.9492493, 4.8059012)
  Uzes → (44.0121279, 4.4196718)
  Nimes → (43.8374249, 4.3600687)
  Aigues Mortes → (43.

,id,city,latitude,longitude
0,1,Mont Saint Michel,48.635954,-1.511460
1,2,St Malo,48.649518,-2.026041
2,3,Bayeux,49.276462,-0.702474
3,4,Le Havre,49.493898,0.107973
4,5,Rouen,49.440459,1.093966
5,6,Paris,48.853495,2.348391
6,7,Amiens,49.894171,2.295695
7,8,Lille,50.636565,3.063528
8,9,Strasbourg,48.584614,7.750713
9,10,Chateau du Haut Koenigsbourg,48.249382,7.343941


## 2. Collecte des données météo / OpenWeatherMap API

On utilise l'endpoint **`/data/2.5/forecast`** de l'API OpenWeatherMap pour récupérer les prévisions sur 5 jours (par blocs de 3h) pour chaque ville.

**Calcul du score météo :**
On construit un score composite pour classer les destinations :
```
weather_score = avg_temp - total_rain - (avg_humidity / 10)
```
- On favorise les **températures élevées**
- On pénalise la **pluie** et l'**humidité**


In [5]:
def get_weather(city_id, city_name, lat, lon):
    url = 'https://api.openweathermap.org/data/2.5/forecast'
    params = {
        'lat': lat, 'lon': lon,
        'appid': OWM_API_KEY,
        'units': 'metric',
        'lang': 'fr'
    }
    response = requests.get(url, params=params)
    data = response.json()

    if 'list' not in data:
        print(f'  Erreur pour {city_name}: {data}')
        return []

    results = []
    for forecast in data['list']:
        results.append({
            'city_id': city_id,
            'city': city_name,
            'datetime': forecast['dt_txt'],
            'temp': forecast['main']['temp'],
            'temp_min': forecast['main']['temp_min'],
            'temp_max': forecast['main']['temp_max'],
            'humidity': forecast['main']['humidity'],
            'weather': forecast['weather'][0]['description'],
            'wind_speed': forecast['wind']['speed'],
            'rain': forecast.get('rain', {}).get('3h', 0)
        })
    return results

all_weather = []
for _, row in df_cities.iterrows():
    print(f'  Météo pour {row["city"]}...')
    weather_data = get_weather(row['id'], row['city'], row['latitude'], row['longitude'])
    all_weather.extend(weather_data)
    time.sleep(1)

df_weather_raw = pd.DataFrame(all_weather)

# Calcul du score météo par ville
df_score = df_weather_raw.groupby(['city_id', 'city']).agg(
    avg_temp=('temp', 'mean'),
    avg_temp_max=('temp_max', 'mean'),
    total_rain=('rain', 'sum'),
    avg_humidity=('humidity', 'mean'),
    avg_wind=('wind_speed', 'mean')
).reset_index()

df_score['weather_score'] = (
    df_score['avg_temp']
    - df_score['total_rain']
    - (df_score['avg_humidity'] / 10)
)
df_score = df_score.sort_values('weather_score', ascending=False)

df_weather_raw.to_csv('weather_raw.csv', index=False)
df_score.to_csv('weather_score.csv', index=False)

print('\nTop 10 destinations météo :')
df_score[['city', 'avg_temp', 'total_rain', 'weather_score']].head(10)


  Météo pour Mont Saint Michel...
  Météo pour St Malo...
  Météo pour Bayeux...
  Météo pour Le Havre...
  Météo pour Rouen...
  Météo pour Paris...
  Météo pour Amiens...
  Météo pour Lille...
  Météo pour Strasbourg...
  Météo pour Chateau du Haut Koenigsbourg...
  Météo pour Colmar...
  Météo pour Eguisheim...
  Météo pour Besancon...
  Météo pour Dijon...
  Météo pour Annecy...
  Météo pour Grenoble...
  Météo pour Lyon...
  Météo pour Gorges du Verdon...
  Météo pour Bormes les Mimosas...
  Météo pour Cassis...
  Météo pour Marseille...
  Météo pour Aix en Provence...
  Météo pour Avignon...
  Météo pour Uzes...
  Météo pour Nimes...
  Météo pour Aigues Mortes...
  Météo pour Saintes Maries de la mer...
  Météo pour Collioure...
  Météo pour Carcassonne...
  Météo pour Ariege...
  Météo pour Toulouse...
  Météo pour Montauban...
  Météo pour Biarritz...
  Météo pour Bayonne...
  Météo pour La Rochelle...

Top 10 destinations météo :


,city,avg_temp,total_rain,weather_score
34,La Rochelle,13.42700,6.96,-1.79550
3,Le Havre,12.06525,10.99,-7.06725
7,Lille,11.74800,12.12,-8.20200
18,Bormes les Mimosas,14.98925,16.07,-8.54325
30,Toulouse,14.01975,16.01,-9.90025
8,Strasbourg,14.35400,17.09,-10.42850
31,Montauban,14.20775,19.09,-12.88725
27,Collioure,15.85900,21.52,-13.40850
26,Saintes Maries de la mer,16.18725,22.84,-14.25775
2,Bayeux,11.36975,17.88,-15.04275


## 3. Collecte des données hôtelières / SerpAPI

SerpAPI interroge Google Hotels pour récupérer les hôtels disponibles dans chaque ville.

### Pourquoi SerpAPI plutôt que Booking.com directement ?

- **Booking.com bloque le scraping** : Cloudflare, CAPTCHA et détection de bots rendent l'extraction directe impossible de manière fiable
- **Overpass API (OpenStreetMap)** a également été testée mais les serveurs publics retournaient des erreurs 406 (surcharge), rendant la collecte instable
- **SerpAPI** fournit une interface stable et structurée vers Google Hotels, avec des données fiables : nom, score, description, coordonnées GPS, lien direct
- Plan gratuit suffisant pour le projet : 100 requêtes/mois offertes, 35 villes = 35 requêtes

On récupère les résultats via le moteur `google_hotels` de SerpAPI, avec filtrage sur la France.

### Points techniques notables

- Clé API gérée via les **Secrets Colab** (`SERPAPI_KEY`), jamais en dur dans le code
- Sleep de 2s entre chaque ville pour respecter les bonnes pratiques
- Les résultats incluent le champ `overall_rating` utilisé comme score hôtelier


In [12]:
!pip install -q google-search-results

from serpapi import GoogleSearch

  Preparing metadata (setup.py) ... done


In [15]:
from serpapi import GoogleSearch

SERPAPI_KEY = userdata.get('SERPAPI_KEY')

params = {
    "engine": "google_hotels",
    "q": "hotels Paris France",
    "api_key": SERPAPI_KEY,
    "hl": "fr",
    "gl": "fr",
    "currency": "EUR"
}

search = GoogleSearch(params)
results = search.get_dict()

# Voir ce qu'on reçoit
print("Clés disponibles dans la réponse :", results.keys())
print("\nExtrait :", str(results)[:1000])

Clés disponibles dans la réponse : dict_keys(['error'])

Extrait : {'error': 'Missing `check_in_date` parameter.'}


In [17]:
from datetime import datetime, timedelta

check_in = (datetime.now() + timedelta(days=30)).strftime("%Y-%m-%d")
check_out = (datetime.now() + timedelta(days=31)).strftime("%Y-%m-%d")

def get_hotels(city_name):
    params = {
        "engine": "google_hotels",
        "q": f"hotels {city_name} France",
        "api_key": SERPAPI_KEY,
        "hl": "fr",
        "gl": "fr",
        "currency": "EUR",
        "check_in_date": check_in,
        "check_out_date": check_out
    }

    try:
        search = GoogleSearch(params)
        results = search.get_dict()
        hotels = []

        for hotel in results.get("properties", []):
            hotels.append({
                "city": city_name,
                "hotel_name": hotel.get("name"),
                "score": hotel.get("overall_rating", None),
                "reviews": hotel.get("reviews", None),
                "description": ", ".join(hotel.get("amenities", [])) if hotel.get("amenities") else "Non renseigné",
                "price_per_night": hotel.get("rate_per_night", {}).get("extracted_lowest", None),
                "url": hotel.get("serpapi_property_details_link", None),
                "latitude": hotel.get("gps_coordinates", {}).get("latitude", None),
                "longitude": hotel.get("gps_coordinates", {}).get("longitude", None)
            })
        return hotels

    except Exception as e:
        print(f"  Erreur {city_name}: {e}")
        return []


# Boucle principale
all_hotels = []
for _, row in df_cities.iterrows():
    print(f"  Hôtels pour {row['city']}...")
    hotels = get_hotels(row["city"])
    print(f"    → {len(hotels)} hôtels trouvés")
    all_hotels.extend(hotels)
    time.sleep(2)

df_hotels_raw = pd.DataFrame(all_hotels)
df_hotels_raw.to_csv("hotels_raw.csv", index=False)
print(f"\n{len(df_hotels_raw)} hôtels bruts collectés")
df_hotels_raw.head()

  Hôtels pour Mont Saint Michel...
    → 20 hôtels trouvés
  Hôtels pour St Malo...
    → 20 hôtels trouvés
  Hôtels pour Bayeux...
    → 20 hôtels trouvés
  Hôtels pour Le Havre...
    → 20 hôtels trouvés
  Hôtels pour Rouen...
    → 20 hôtels trouvés
  Hôtels pour Paris...
    → 20 hôtels trouvés
  Hôtels pour Amiens...
    → 20 hôtels trouvés
  Hôtels pour Lille...
    → 20 hôtels trouvés
  Hôtels pour Strasbourg...
    → 20 hôtels trouvés
  Hôtels pour Chateau du Haut Koenigsbourg...
    → 20 hôtels trouvés
  Hôtels pour Colmar...
    → 20 hôtels trouvés
  Hôtels pour Eguisheim...
    → 20 hôtels trouvés
  Hôtels pour Besancon...
    → 20 hôtels trouvés
  Hôtels pour Dijon...
    → 20 hôtels trouvés
  Hôtels pour Annecy...
    → 20 hôtels trouvés
  Hôtels pour Grenoble...
    → 20 hôtels trouvés
  Hôtels pour Lyon...
    → 20 hôtels trouvés
  Hôtels pour Gorges du Verdon...
    → 20 hôtels trouvés
  Hôtels pour Bormes les Mimosas...
    → 20 hôtels trouvés
  Hôtels pour Cassis...
 

,city,hotel_name,score,reviews,description,price_per_night,url,latitude,longitude
0,Mont Saint Michel,Auberge de la Baie - Hôtel Mont Saint-Michel,4.1,1118.0,"Breakfast ($), Free Wi-Fi, Free parking, Air c...",77.0,https://serpapi.com/search.json?adults=2&check...,48.615812,-1.488360
1,Mont Saint Michel,Le Mouton Blanc,3.3,1931.0,"Breakfast, Wi-Fi, Pet-friendly, Restaurant, Ki...",191.0,https://serpapi.com/search.json?adults=2&check...,48.635780,-1.509744
2,Mont Saint Michel,ibis Pontorson Baie du Mont-Saint-Michel,4.2,2157.0,"Breakfast ($), Free Wi-Fi, Free parking, Outdo...",113.0,https://serpapi.com/search.json?adults=2&check...,48.568115,-1.542957
3,Mont Saint Michel,Hôtel Mercure Mont-Saint-Michel,4.2,2250.0,"Breakfast ($), Free Wi-Fi, Free parking, Air c...",163.0,https://serpapi.com/search.json?adults=2&check...,48.614335,-1.510680
4,Mont Saint Michel,ibis budget Pontorson Baie du Mont-Saint-Michel,4.7,22.0,"Breakfast ($), Free Wi-Fi, Free parking, Air c...",80.0,https://serpapi.com/search.json?adults=2&check...,48.568119,-1.542956


## 4. Nettoyage des données hôtelières

Avant de stocker en Data Warehouse, on nettoie le dataset :
- Suppression des hôtels sans coordonnées GPS
- Conversion du score en numérique
- Remplissage des valeurs manquantes (médiane pour les scores)
- Déduplication sur (nom, ville)
- Ajout d'un identifiant unique


In [19]:
df_hotels = df_hotels_raw.copy()
print(f'Hôtels bruts : {len(df_hotels)}')

# Suppression sans GPS
df_hotels = df_hotels.dropna(subset=['latitude', 'longitude'])
print(f'Après suppression sans GPS : {len(df_hotels)}')

# Conversion score en numérique
df_hotels['score'] = pd.to_numeric(df_hotels['score'], errors='coerce')

# Valeurs manquantes
df_hotels['description'] = df_hotels['description'].fillna('Non renseigné')
df_hotels['score'] = df_hotels['score'].fillna(df_hotels['score'].median())

# Déduplication
df_hotels = df_hotels.drop_duplicates(subset=['hotel_name', 'city'])
print(f'Après déduplication : {len(df_hotels)}')

# Identifiant unique
df_hotels = df_hotels.reset_index(drop=True)
df_hotels['hotel_id'] = df_hotels.index + 1

# Réorganisation colonnes (sans city_id qui n'existe plus)
df_hotels = df_hotels[['hotel_id', 'city', 'hotel_name',
                        'score', 'reviews', 'price_per_night',
                        'description', 'url', 'latitude', 'longitude']]

df_hotels.to_csv('hotels_clean.csv', index=False)
print(f'\nDataset final : {len(df_hotels)} hôtels')
df_hotels.head()


Hôtels bruts : 700
Après suppression sans GPS : 700
Après déduplication : 700

Dataset final : 700 hôtels


,hotel_id,city,hotel_name,score,reviews,price_per_night,description,url,latitude,longitude
0,1,Mont Saint Michel,Auberge de la Baie - Hôtel Mont Saint-Michel,4.1,1118.0,77.0,"Breakfast ($), Free Wi-Fi, Free parking, Air c...",https://serpapi.com/search.json?adults=2&check...,48.615812,-1.488360
1,2,Mont Saint Michel,Le Mouton Blanc,3.3,1931.0,191.0,"Breakfast, Wi-Fi, Pet-friendly, Restaurant, Ki...",https://serpapi.com/search.json?adults=2&check...,48.635780,-1.509744
2,3,Mont Saint Michel,ibis Pontorson Baie du Mont-Saint-Michel,4.2,2157.0,113.0,"Breakfast ($), Free Wi-Fi, Free parking, Outdo...",https://serpapi.com/search.json?adults=2&check...,48.568115,-1.542957
3,4,Mont Saint Michel,Hôtel Mercure Mont-Saint-Michel,4.2,2250.0,163.0,"Breakfast ($), Free Wi-Fi, Free parking, Air c...",https://serpapi.com/search.json?adults=2&check...,48.614335,-1.510680
4,5,Mont Saint Michel,ibis budget Pontorson Baie du Mont-Saint-Michel,4.7,22.0,80.0,"Breakfast ($), Free Wi-Fi, Free parking, Air c...",https://serpapi.com/search.json?adults=2&check...,48.568119,-1.542956


## 5. Data Lake : Stockage sur AWS S3

On uploade les fichiers bruts et nettoyés dans un **bucket S3** (Simple Storage Service).

S3 joue le rôle de **Data Lake** : stockage centralisé, brut, non structuré, accessible depuis n'importe quel service AWS.

Les fichiers sont organisés dans un préfixe `raw/` pour suivre les bonnes pratiques de partitionnement.


In [20]:
s3 = boto3.client(
    's3',
    region_name='eu-west-3',
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET
)

files_to_upload = [
    'cities_gps.csv',
    'weather_raw.csv',
    'weather_score.csv',
    'hotels_raw.csv',
    'hotels_clean.csv'
]

for filepath in files_to_upload:
    filename = os.path.basename(filepath)
    print(f'  Upload de {filename}...')
    try:
        s3.upload_file(filepath, BUCKET_NAME, f'raw/{filename}')
        print(f'  ✓ {filename} → s3://{BUCKET_NAME}/raw/{filename}')
    except Exception as e:
        print(f'  ✗ Erreur : {e}')

print('\nTous les fichiers sont dans le Data Lake S3 !')


  Upload de cities_gps.csv...
  ✓ cities_gps.csv → s3://kayak-project-marine/raw/cities_gps.csv
  Upload de weather_raw.csv...
  ✓ weather_raw.csv → s3://kayak-project-marine/raw/weather_raw.csv
  Upload de weather_score.csv...
  ✓ weather_score.csv → s3://kayak-project-marine/raw/weather_score.csv
  Upload de hotels_raw.csv...
  ✓ hotels_raw.csv → s3://kayak-project-marine/raw/hotels_raw.csv
  Upload de hotels_clean.csv...
  ✓ hotels_clean.csv → s3://kayak-project-marine/raw/hotels_clean.csv

Tous les fichiers sont dans le Data Lake S3 !


## 6. Data Warehouse : ETL vers AWS RDS MySQL

On charge les données nettoyées dans un **Data Warehouse** (RDS MySQL) via un pipeline ETL :

- **Extract** : lecture des CSV depuis S3
- **Transform** : nettoyage colonnes, arrondi des valeurs numériques
- **Load** : insertion dans 3 tables relationnelles via SQLAlchemy

RDS est différent de S3 : c'est une **base relationnelle structurée**, optimisée pour les requêtes SQL analytiques.


In [23]:
# --- EXTRACT : lecture depuis S3 ---
print('Lecture des données depuis S3...')

def read_csv_from_s3(filename):
    response = s3.get_object(Bucket=BUCKET_NAME, Key=f'raw/{filename}')
    content = response['Body'].read().decode('utf-8')
    return pd.read_csv(io.StringIO(content))

df_cities_etl  = read_csv_from_s3('cities_gps.csv')
df_weather_etl = read_csv_from_s3('weather_score.csv')
df_hotels_etl  = read_csv_from_s3('hotels_clean.csv')

print(f'  Villes  : {len(df_cities_etl)} lignes')
print(f'  Météo   : {len(df_weather_etl)} lignes')
print(f'  Hôtels  : {len(df_hotels_etl)} lignes')

# --- TRANSFORM ---
print('\nTransformation...')

for df in [df_cities_etl, df_weather_etl, df_hotels_etl]:
    df.columns = df.columns.str.lower().str.replace(' ', '_')

for col in ['avg_temp', 'avg_temp_max', 'total_rain', 'avg_humidity', 'weather_score']:
    if col in df_weather_etl.columns:
        df_weather_etl[col] = df_weather_etl[col].round(2)

df_hotels_etl['latitude']  = df_hotels_etl['latitude'].round(6)
df_hotels_etl['longitude'] = df_hotels_etl['longitude'].round(6)

# --- LOAD : insertion dans RDS ---
print('\nConnexion à RDS...')

engine = create_engine(
    f'mysql+pymysql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/'
)
with engine.connect() as conn:
    conn.execute(text(f'CREATE DATABASE IF NOT EXISTS {RDS_DB}'))
    print(f"  Base '{RDS_DB}' créée")

engine = create_engine(
    f'mysql+pymysql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DB}'
)

df_cities_etl.to_sql('cities',  engine, if_exists='replace', index=False)
df_weather_etl.to_sql('weather', engine, if_exists='replace', index=False)
df_hotels_etl.to_sql('hotels',  engine, if_exists='replace', index=False)
print('  Tables cities / weather / hotels chargées')

# --- VÉRIFICATION SQL ---
print('\nVérification avec requêtes SQL...')

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT c.city, w.avg_temp, w.total_rain, w.weather_score
        FROM weather w
        JOIN cities c ON w.city_id = c.id
        ORDER BY w.weather_score DESC
        LIMIT 5
    """))
    print('\nTop 5 destinations météo :')
    for row in result:
        print(f'  {row[0]} → score: {row[3]:.2f}, temp: {row[1]:.1f}°C, pluie: {row[2]:.1f}mm')

    result = conn.execute(text("""
        SELECT city, COUNT(*) as nb_hotels
        FROM hotels
        GROUP BY city
        ORDER BY nb_hotels DESC
        LIMIT 5
    """))
    print('\nTop 5 villes par nombre d\'hôtels :')
    for row in result:
        print(f'  {row[0]} → {row[1]} hôtels')


Lecture des données depuis S3...
  Villes  : 35 lignes
  Météo   : 35 lignes
  Hôtels  : 700 lignes

Transformation...

Connexion à RDS...
  Base 'kayak' créée
  Tables cities / weather / hotels chargées

Vérification avec requêtes SQL...

Top 5 destinations météo :
  La Rochelle → score: -1.80, temp: 13.4°C, pluie: 7.0mm
  Le Havre → score: -7.07, temp: 12.1°C, pluie: 11.0mm
  Lille → score: -8.20, temp: 11.8°C, pluie: 12.1mm
  Bormes les Mimosas → score: -8.54, temp: 15.0°C, pluie: 16.1mm
  Toulouse → score: -9.90, temp: 14.0°C, pluie: 16.0mm

Top 5 villes par nombre d'hôtels :
  Nimes → 20 hôtels
  Avignon → 20 hôtels
  Uzes → 20 hôtels
  Aix en Provence → 20 hôtels
  Aigues Mortes → 20 hôtels


## 7. Visualisations / Cartes interactives Plotly

On produit deux cartes interactives pour l'application finale :
1. **Top 5 destinations météo** : classement par score météo composite
2. **Top 20 hôtels** dans les meilleures destinations : classement par nombre d'étoiles


In [25]:
# Jointure météo + coordonnées GPS
df_map = df_score.merge(df_cities[['id', 'latitude', 'longitude']],
                        left_on='city_id', right_on='id')

# Normalisation du score pour la taille (doit être positif)
df_map['score_size'] = df_map['weather_score'] - df_map['weather_score'].min() + 1

# --- CARTE 1 : Top 5 destinations météo ---
df_top5 = df_map.sort_values('weather_score', ascending=False).head(5)

fig1 = px.scatter_map(
    df_top5,
    lat='latitude', lon='longitude',
    size='score_size',
    color='avg_temp',
    hover_name='city',
    hover_data={
        'avg_temp': ':.1f',
        'total_rain': ':.1f',
        'weather_score': ':.2f',
        'score_size': False
    },
    color_continuous_scale='RdYlGn',
    size_max=40, zoom=4,
    center={'lat': 46.5, 'lon': 2.5},
    map_style='carto-positron',
    title='Top 5 destinations météo en France'
)
fig1.update_layout(margin={'r':0,'t':40,'l':0,'b':0})
fig1.write_html('map_top5_destinations.html')
fig1.show()
print('Carte 1 OK')


Carte 1 OK


In [26]:
# --- CARTE 2 : Top 20 hôtels dans les meilleures destinations ---
top5_cities = df_top5['city'].tolist()
df_top_hotels = df_hotels[df_hotels['city'].isin(top5_cities)].copy()
df_top_hotels['score'] = pd.to_numeric(df_top_hotels['score'], errors='coerce')
df_top20 = df_top_hotels.dropna(subset=['score']).sort_values('score', ascending=False).head(20)

# Normalisation du score pour la taille
df_top20 = df_top20.copy()
df_top20['score_size'] = df_top20['score'] - df_top20['score'].min() + 1

fig2 = px.scatter_map(
    df_top20,
    lat='latitude', lon='longitude',
    size='score_size', color='score',
    hover_name='hotel_name',
    hover_data={
        'city': True,
        'score': ':.1f',
        'description': True,
        'score_size': False
    },
    color_continuous_scale='Blues',
    size_max=30, zoom=4,
    center={'lat': 46.5, 'lon': 2.5},
    map_style='carto-positron',
    title='Top 20 hôtels des meilleures destinations'
)
fig2.update_layout(margin={'r':0,'t':40,'l':0,'b':0})
fig2.write_html('map_top20_hotels.html')
fig2.show()
print('Carte 2 OK')

Carte 2 OK


## 8. Discussion Passage à l'échelle (Big Data)

À cette échelle (35 villes, 700 hôtels), **Pandas et MySQL suffisent largement**. Mais si on passait à l'échelle mondiale — des milliers de villes, des millions d'hôtels, des données en temps réel — il faudrait adapter l'architecture :

| Besoin | Outil actuel | Outil à l'échelle |
|--------|-------------|-------------------|
| Traitement distribué | Pandas | **Apache Spark** (PySpark) |
| Data Warehouse analytique | RDS MySQL | **Amazon Redshift** (colonnaire) |
| Exécution Spark managée | Local | **Amazon EMR** |
| Ingestion temps réel | Batch manuel | **Apache Kafka** |

L'**architecture resterait la même** (Sources → Data Lake → ETL → Warehouse → Visu), seuls les outils changeraient pour absorber le volume et la vélocité des données.


## 9. Conclusion

Ce projet a permis de construire un pipeline de données complet :

| Étape | Outil | Résultat |
|-------|-------|----------|
| Géocodage | Nominatim | 35 villes avec coordonnées GPS |
| Météo | OpenWeatherMap API | Prévisions 5 jours + score météo |
| Hôtels | SerpAPI (Google Hotels) | 700 hôtels collectés (20/ville) |
| Data Lake | AWS S3 | 3 CSV bruts stockés |
| Data Warehouse | AWS RDS MySQL | 3 tables relationnelles nettoyées |
| Visualisations | Plotly | 2 cartes interactives |

**Bonnes pratiques appliquées :**
- Credentials gérés via Secrets Colab (jamais en dur dans le code)
- Rate limiting respecté sur toutes les APIs
- Séparation Data Lake (brut) / Data Warehouse (nettoyé)
- Vérification du pipeline via requêtes SQL avec JOINs
- Limite de 20 hôtels/ville pour rester dans le quota gratuit SerpAPI
